In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import rootutils

In [3]:
rootutils.setup_root(
    os.getcwd(), indicator=".project-root", dotenv=True, pythonpath=True, cwd=False
)

PosixPath('/data/gena-lm-mk-2/old_gena_lm/downstream_tasks/caduceus')

In [4]:
name = "kuleshov-group/caduceus-ph_seqlen-131k_d_model-256_n_layer-16"

In [5]:
import transformers

/data/gena-lm-mk-2/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
caduceus = transformers.AutoModel.from_pretrained(name, trust_remote_code=True)

In [7]:
caduceus

Caduceus(
  (backbone): CaduceusMixerModel(
    (embeddings): CaduceusEmbeddings(
      (word_embeddings): Embedding(16, 256)
    )
    (layers): ModuleList(
      (0-15): 16 x Block(
        (norm): RMSNorm()
        (mixer): BiMambaWrapper(
          (mamba_fwd): Mamba(
            (in_proj): Linear(in_features=256, out_features=1024, bias=False)
            (conv1d): Conv1d(512, 512, kernel_size=(4,), stride=(1,), padding=(3,), groups=512)
            (act): SiLU()
            (x_proj): Linear(in_features=512, out_features=48, bias=False)
            (dt_proj): Linear(in_features=16, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=256, bias=False)
          )
          (mamba_rev): Mamba(
            (in_proj): Linear(in_features=256, out_features=1024, bias=False)
            (conv1d): Conv1d(512, 512, kernel_size=(4,), stride=(1,), padding=(3,), groups=512)
            (act): SiLU()
            (x_proj): Linear(in_features=512, out_feature

In [8]:
import hydra

In [17]:
!export GENALM_HOME=/data/genalm

In [9]:
os.environ["GENALM_HOME"]="/data/genalm"

In [10]:
with hydra.initialize(version_base=None, config_path="configs/"):
    cfg = hydra.compose(config_name="v2_qnorm_large_mouse.yaml")

In [11]:
dataset = hydra.utils.instantiate(cfg.training_dataset)

In [16]:
sample = dataset[335]
print(sorted(sample.keys()))

['chrom', 'dataset_description', 'desc_vectors', 'end', 'gene_id', 'input_ids', 'labels', 'labels_mask', 'name', 'reverse', 'selected_keys', 'start']


In [ ]:
sample["labels"].shape, sample["labels_mask"].shape, sample["input_ids"].shape, sample["desc_vectors"].shape

In [ ]:
from model.model import ExpressionCountsModel

In [ ]:
from tqdm.autonotebook import tqdm

In [ ]:
collator = hydra.utils.instantiate(cfg.collate_fn)

In [ ]:
import torch

In [ ]:
dataloader = torch.utils.data.DataLoader(
    dataset = dataset,
    num_workers = 1,
    shuffle = False,
    batch_size = 2,
    collate_fn = collator
)

In [ ]:
for i, cpu_batch in enumerate(tqdm(dataloader)):
    cuda_batch = dict()
    for key, value in cpu_batch.items():
        if torch.is_tensor(value):
            cuda_batch[key] = value.to(device = "cuda")
        else:
            cuda_batch[key] = value
    if 16 < i:
        break

In [ ]:
cuda_batch["input_ids"]

In [ ]:
cfg_model = hydra.utils.instantiate(cfg.model).to(device = "cuda")

In [ ]:
result = cfg_model(**cuda_batch)

In [ ]:
result